In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer 
from sklearn.model_selection import StratifiedKFold




In [3]:
#  create The Custom Kaggle NDCG metric
def calculate_ndcg5(y_true, y_pred_proba):
    """
    Calculates the exact NDCG@5 score used by Kaggle.
    y_true: 1D array of true encoded labels
    y_pred_proba: 2D array of model probabilities
    """
    # Get the column index (class) of the top 5 highest probabilities per user
    top5_preds = np.argsort(y_pred_proba, axis=1)[:, :-6:-1]
    
    # Check which of the 5 positions matches the true label
    matches = (top5_preds == y_true.reshape(-1, 1))
    
    # Apply the discount math: 1 / log2(rank + 1)
    discounts = 1 / np.log2(np.arange(1, 6) + 1)
    
    # Multiply matches by discounts and get the average score across all users
    ndcg_scores = np.sum(matches * discounts, axis=1)
    return np.mean(ndcg_scores)

In [4]:

df = pd.read_csv('final_train.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 213451 entries, 0 to 213450
Data columns (total 29 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       213451 non-null  str    
 1   gender                   213451 non-null  str    
 2   age                      213451 non-null  float64
 3   signup_method            213451 non-null  str    
 4   signup_flow              213451 non-null  str    
 5   language                 213451 non-null  str    
 6   affiliate_channel        213451 non-null  str    
 7   affiliate_provider       213451 non-null  str    
 8   first_affiliate_tracked  213451 non-null  str    
 9   signup_app               213451 non-null  str    
 10  first_browser            213451 non-null  str    
 11  country_destination      213451 non-null  str    
 12  is_train                 213451 non-null  bool   
 13  total_actions            213451 non-null  float64
 14  total_seconds  

In [5]:
X = df.drop(['id', 'country_destination', 'is_train'], axis=1)
y = df['country_destination']

print("2. Encoding the Target...")
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Save the mapping 
target_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Target Mapping: {target_mapping}")


2. Encoding the Target...
Target Mapping: {'AU': np.int64(0), 'CA': np.int64(1), 'DE': np.int64(2), 'ES': np.int64(3), 'FR': np.int64(4), 'GB': np.int64(5), 'IT': np.int64(6), 'NDF': np.int64(7), 'NL': np.int64(8), 'PT': np.int64(9), 'US': np.int64(10), 'other': np.int64(11)}


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded    
)
X_train.shape, X_val.shape, y_train.shape, y_val.shape   

((170760, 26), (42691, 26), (170760,), (42691,))

In [7]:
numerical_cols = X.select_dtypes(include=['int64', 'float64' ]).columns
categorical_cols = X.select_dtypes(include=['object' , 'str']).columns 

In [8]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [9]:

x_train_preprocessed =preprocessor.fit_transform(X_train, y_train)
x_val_preprocessed = preprocessor.transform(X_val) 

### RF with train_test_split 

In [14]:

rf = RandomForestClassifier(n_estimators=500, min_samples_split=5, min_samples_leaf=1, max_depth=25, class_weight='balanced', random_state=42)
rf.fit(x_train_preprocessed, y_train)

pred_y_train = rf.predict_proba(x_train_preprocessed)
train_ndcg = calculate_ndcg5(y_train, pred_y_train)
print(f"RF Training NDCG@5: {train_ndcg:.5f}")


pred_y_val = rf.predict_proba(x_val_preprocessed)
val_ndcg = calculate_ndcg5(y_val, pred_y_val)
print(f"RF Validation NDCG@5: {val_ndcg:.5f}")


RF Training NDCG@5: 0.94998
RF Validation NDCG@5: 0.79275


### RF with Stratified k-fold & RandomSearch  

In [11]:
x_preprocessed = preprocessor.fit_transform(X)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer

custom_ndcg_scorer = make_scorer(calculate_ndcg5, response_method='predict_proba')

rf = RandomForestClassifier( ,class_weight='balanced', random_state=42)

params = {
    'n_estimators': [300, 500],       
    'max_depth': [20, 25],       
    'min_samples_split': [5,10],
    'min_samples_leaf': [1, 2]    
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=params,
    n_iter=10,                
    scoring=custom_ndcg_scorer, 
    cv=skf,                  
    n_jobs=2,
    random_state=42 , 
    error_score='raise'
)

random_search.fit(x_preprocessed, y_encoded)

print(f"\nBest Hyperparameters Found:\n{random_search.best_params_}")


best_rf = random_search.best_estimator_

pred_y_train = best_rf.predict_proba(x_preprocessed)
train_ndcg = calculate_ndcg5(y_encoded, pred_y_train)


cv_ndcg = random_search.best_score_

print(f"Best RF Train NDCG@5: {train_ndcg:.5f}")
print(f"Best RF CV NDCG@5:    {cv_ndcg:.5f}")


Best Hyperparameters Found:
{'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 25}
Best RF Train NDCG@5: 0.94387
Best RF CV NDCG@5:    0.79576


Note -> Random Forest didn't learn the patterns ; it memorized the training dataset   , its performance crashed down to 79% on validation data .   -> this is **overfitting** 
